<a href="https://colab.research.google.com/github/SaraLarkem/ireland-fuel-grocery-viz/blob/main/notebooks/Data_Cleaning_Subcategories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook: Data_Cleaning_Subcategories.ipynb

**Goal:** Ingest and standardize the raw Consumer Price Index (CPI) item data from CSO Ireland.

The raw data is correctly formatted but not yet optimized for time-series analysis. This notebook converts string-based dates (e.g., "2016 January") into searchable datetime objects, removes redundant categorical metadata, and prepares the clean file required for Avery's subcategory analysis (Notebook 05).


In [3]:
import pandas as pd

# Load the raw data directly from the project GitHub repository
url = 'https://raw.githubusercontent.com/SaraLarkem/ireland-fuel-grocery-viz/refs/heads/main/data/raw_perishable_non_perishable_data.csv'
df = pd.read_csv(url)

# Display initial state
print("--- Initial Raw Data ---")
print(df.head(3))

--- Initial Raw Data ---
          STATISTIC Label         Month  \
0  National Average Price  2016 January   
1  National Average Price  2016 January   
2  National Average Price  2016 January   

                           Consumer Item  UNIT  VALUE  
0     White, self raising flour per 2 kg  Euro  2.378  
1        Brown, wholemeal flour per 2 kg  Euro  3.033  
2  Bread, white sliced pan, large (800g)  Euro  1.263  


### 1. Data Standardization and Type Conversion
To perform rolling averages and lag correlations later in the pipeline, the `Date` column must be a formal Pandas Datetime object rather than plain text. We will also strip out the `STATISTIC Label` and `UNIT` columns, as every row identically represents the "National Average Price" in "Euros", making those columns redundant for programmatic analysis.

In [4]:
# 1. Rename columns for easier access and coding consistency
df = df.rename(columns={
    'STATISTIC Label': 'Statistic',
    'Month': 'Date',
    'Consumer Item': 'Item',
    'UNIT': 'Unit',
    'VALUE': 'Price'
})

# 2. Convert 'Date' string (e.g., "2016 January") to a proper datetime object
df['Date'] = pd.to_datetime(df['Date'], format='%Y %B')

# 3. Drop unnecessary columns
df = df.drop(columns=['Statistic', 'Unit'])

# 4. Sort chronologically and alphabetically by item to ensure a clean index
df = df.sort_values(by=['Date', 'Item']).reset_index(drop=True)

# 5. Basic validation check to confirm cleaning success
print(f"Cleaned dataset range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total rows: {len(df)}")
print("\n--- Cleaned Data Sample ---")
print(df.head())

Cleaned dataset range: 2016-01-01 to 2026-01-01
Total rows: 1694

--- Cleaned Data Sample ---
        Date                                   Item  Price
0 2016-01-01  Bread, white sliced pan, large (800g)  1.263
1 2016-01-01        Brown, wholemeal flour per 2 kg  3.033
2 2016-01-01                          Butter per lb  2.727
3 2016-01-01              Full fat milk per 2 litre  1.714
4 2016-01-01                             Jam per lb  1.383


### 2. Export to Pipeline
With the dates properly formatted and the columns standardized, this clean dataset is exported to the `data/` directory.
* **(Notebook 05)** will use this output to pivot the items into perishable (meat, dairy, fresh produce) vs. shelf-stable (cereals, bread) groups to analyze differing reactions to fuel spikes.

In [5]:
import os

# Ensure the data directory exists before exporting
os.makedirs('data', exist_ok=True)

# Save the cleaned dataset for Notebook 05
export_path = 'data/cleaned_grocery_subcategories.csv'
df.to_csv(export_path, index=False)

print(f"Cleaning complete. Data exported to: {export_path}")

Cleaning complete. Data exported to: data/cleaned_grocery_subcategories.csv
